In [8]:
from ollama import chat

response = chat(
    model='llama3.2',
    messages=[
        {
            'role': 'user',
            'content': 'Explique o que é RAG em uma frase.'
        }
    ]
)

print(response['message']['content'])

RAG é um termo inglês que significa "Reproduction Assistance Group" ou "Reprodução Assistida", referindo-se a um processo de biologia assistida que envolve o tratamento de doenças genéticas hereditárias.


# Projeto Final - Sistemas Cognitivos com Large Language Models

## Tema

Assistente Inteligente para Análise de Avaliações de Clientes de E-commerce

## Objetivo

Construir um sistema baseado em LLM capaz de analisar avaliações de clientes, identificar problemas relatados e gerar respostas estruturadas utilizando técnicas de Prompt Engineering, Embeddings e RAG.

In [1]:
import pandas as pd
import chromadb
import ollama
from sentence_transformers import SentenceTransformer

In [11]:
import sys
print(sys.executable)

c:\Python311\python.exe


In [2]:
df = pd.read_csv("olist_order_reviews_dataset.csv")

In [3]:
documents = df['review_comment_message'].dropna().tolist()

print("Quantidade de documentos:", len(documents))
print("\nPrimeiro documento:")
print(documents[0])

Quantidade de documentos: 40977

Primeiro documento:
Recebi bem antes do prazo estipulado.


In [4]:
from sentence_transformers import SentenceTransformer
import chromadb

## 2.Carregar modelo de embeddings

In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Modelo carregado!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\T-GAMER\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\T-GAMER\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado!


## 3.Criar banco vetorial 

In [6]:
client = chromadb.Client()

collection = client.create_collection(
    name="reviews_olist"
)

print("Coleção criada!")

Coleção criada!


In [7]:
sample_docs = documents[:1000]

collection.add(
    documents=sample_docs,
    ids=[str(i) for i in range(len(sample_docs))]
)

print("Documentos indexados!")

C:\Users\T-GAMER\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:37<00:00, 2.20MiB/s]  


Documentos indexados!


In [8]:
resultado = collection.query(
    query_texts=["produto chegou atrasado"],
    n_results=3
)

print(resultado["documents"])

[['O produto ainda não chegou', 'O produto chegou rasgado ', 'Produto ótimo. .. chegou antes do esperado.']]


In [9]:
def buscar_contexto(pergunta, k=3):

    resultado = collection.query(
        query_texts=[pergunta],
        n_results=k
    )

    docs = resultado["documents"][0]

    return "\n\n".join(docs)

In [10]:
contexto = buscar_contexto(
    "clientes reclamando de atraso na entrega"
)

print(contexto)

Tenha sempre esse atendimento ao cliente .

Sem pontualidade não da satisfação ao cliente.

Loja em si é boa pena a americana nao da suporte ao cliente 


## 4.Conectar ao Ollama 

In [11]:
import ollama
resposta = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Explique o que é PLN em uma frase."
        }
    ]
)

print(resposta["message"]["content"])

O PLN (Peso Livre Nadais) é um tipo de câmbio que permite ao exchange do Real com qualquer moeda, sem a necessidade de taxas fixas ou margens, oferecendo uma liberdade maior para os investidores e comerciantes.


##5. Pipeline RAG (consulta + recuperação + resposta)

In [12]:
query = "Quais são as principais reclamações dos clientes?"

resultado = collection.query(
    query_texts=[query],
    n_results=5
)

reviews = resultado["documents"][0]

print("Reviews recuperadas:")
for review in reviews:
    print("-", review)

Reviews recuperadas:
- Sem pontualidade não da satisfação ao cliente.
- Tenha sempre esse atendimento ao cliente .
- Vendedor compromisso do vou o cliente
- Cada vez que compro mais fico satisfeita parabéns pela honestidade com seus clientes 👏👏👏👏?
- ESPERO NUNCA MAIS COMPRAR E TER QUE RESPONDER ESSE TIPO DE AVALIAÇÃO


In [13]:
contexto = "\n\n".join(reviews)

print(contexto)

Sem pontualidade não da satisfação ao cliente.

Tenha sempre esse atendimento ao cliente .

Vendedor compromisso do vou o cliente

Cada vez que compro mais fico satisfeita parabéns pela honestidade com seus clientes 👏👏👏👏?

ESPERO NUNCA MAIS COMPRAR E TER QUE RESPONDER ESSE TIPO DE AVALIAÇÃO


In [14]:
prompt = f"""
Você é um analista de experiência do cliente.

Com base nas avaliações abaixo, responda:

1. Quais são as principais reclamações?
2. Quais padrões aparecem?
3. Faça um resumo executivo.

Avaliações:
{contexto}
"""

resposta = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(resposta["message"]["content"])

**Resumo Executivo**

Avaliação do Atendimento ao Cliente:

*   **Reclamações Principais**: Falta de pontualidade, falta de atendimento e comunicação eficaz.
*   **Padrões Aparecidos**: Comportamento incompatível com os padrões esperados por um vendedor comprometido. O cliente expressa satisfação com a honestidade do vendedor, mas não valoriza a pontualidade e o atendimento devido à falta de comunicação eficaz.
*   **Conclusão**: A avaliação destaca a importância da pontualidade e do atendimento eficaz em uma relação com cliente. Embora o vendedor tenha demonstrado honestidade, isso não compensa a falta de comunicação adequada e a falha na cumprir os compromissos.


## 6. Prompt Engineering

1. Zero-shot

In [15]:
prompt_zero = """
Classifique a reclamação abaixo em uma categoria:
- entrega
- produto
- atendimento

Reclamação:
O produto chegou com 15 dias de atraso.

Responda apenas com a categoria.
"""

resposta = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": prompt_zero}]
)

print(resposta["message"]["content"])

entrega


2. Few-shot

In [16]:
prompt_few = """
Exemplos:

Reclamação: Produto chegou quebrado.
Categoria: produto

Reclamação: Demoraram para responder meu e-mail.
Categoria: atendimento

Reclamação: Pedido chegou atrasado.
Categoria: entrega

Agora classifique:

Reclamação: Minha encomenda atrasou 10 dias.
Categoria:
"""

resposta = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": prompt_few}]
)

print(resposta["message"]["content"])

Segundo a categoria indicada, posso dizer que a reclamação deve ser classificada como "Entrega". 

A entrega é uma das categorias mais importantes para os clientes, pois estão sempre preocupados com o tempo de entrega e se há alguma demora ou problema durante a entrega do seu pedido.


3. Chain-of-thought

In [17]:
prompt_cot = """
Analise a reclamação passo a passo.

Reclamação:
Recebi o produto correto, mas a entrega atrasou 12 dias.

Pense passo a passo e explique a categoria.
"""

resposta = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": prompt_cot}]
)

print(resposta["message"]["content"])

Vamos analisar a reclamação passo a passo:

**Passo 1: Identificação do problema**

* O cliente relata que recebeu o produto correto, mas com uma entrega atrasada de 12 dias.

**Passo 2: Análise da situação**

* A empresa precisa entender o problema para identificar se é um erro de envio, problemas de logística ou algo mais complexo.
* Em este caso, parece ser um problema de entrega que não atendeu às expectativas do cliente.

**Passo 3: Categorização do problema**

* Com base na reclamação, podemos categorizar o problema como:
 + Problema de entrega
 + Atraso no envio

Essa categoria é importante porque ajuda a entender a causa raiz do problema e a priorizar as ações a serem tomadas para resolver o assunto.

**Passo 4: Análise das implicações**

* O cliente está satisfeito com o produto, mas não está satisfeito com o tempo de entrega.
* A empresa precisa considerar as implicações do atraso e como pode afetar a satisfação do cliente.

**Passo 5: Planejamento de solução**

* A empresa p